# Scaling and regime geometry audit

Second-stage train-only research audit. Run the script first. This notebook must not load calibration/test data.


In [ ]:
from pathlib import Path
import json
import pandas as pd

summary = json.loads(Path('../docs/research/scaling_regime_geometry.json').read_text())
features = pd.read_parquet('../data/processed/manifold_audit/scaling_regime_feature_diagnostics.parquet')
regimes = pd.read_parquet('../data/processed/manifold_audit/regime_geometry_results.parquet')
summary['data_boundary'], summary['regime_definition']


## Scaling leverage
Inspect whether Robust scaling amplifies small-IQR/heavy-tail features relative to Standard scaling.


In [ ]:
for scaler in ['robust', 'standard']:
    print('\n', scaler.upper())
    display(features[features.scaler == scaler].sort_values('fit_scaled_variance', ascending=False).head(12))


## Global versus regime-conditioned PCA
Comparisons use the same scaler and the same global 95%-variance latent dimension.


In [ ]:
regimes.sort_values(['scaler', 'regime'])


## Neighbor purity and subspace stability
High local regime purity plus smaller within-regime than between-regime angles supports distinct operating manifolds.


In [ ]:
{s: {
    'neighbor_purity': summary['scalers'][s]['neighbor_regime_purity'],
    'subspace': summary['scalers'][s]['subspace'],
} for s in summary['scalers']}


## Interpretation gate

- scaling effect collapses under Standard -> representation/scaling issue first;
- regime PCA wins under both scalers + high neighbor purity -> regime-conditioned model next;
- large within-regime instability remains -> investigate nonlinear/temporal structure inside regimes.
